In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
import mlflow

def get_live_weather(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon,
        "hourly": "temperature_2m,windspeed_10m,cloudcover",
        "timezone": "auto", "forecast_days": 1
    }
    data = requests.get(url, params=params).json()["hourly"]
    return spark.createDataFrame(pd.DataFrame(data))

print("Fetching Paris weather...")
live_df = get_live_weather(48.8566, 2.3522)

live_df = live_df.withColumn("time", F.to_timestamp("time")) \
                 .withColumn("hour", F.hour("time")) \
                 .withColumn("month", F.month("time")) \
                 .withColumn("day_of_week", F.dayofweek("time"))

pipeline = PipelineModel.load("/Volumes/workspace/default/my_volume/weather_pipeline")

volume_temp_path = "/Volumes/workspace/default/my_volume/mlflow_tmp"

model_uri = "models:/workspace.default.power_risk_model_production@champion"

model = mlflow.spark.load_model(model_uri, dfs_tmpdir=volume_temp_path)

print("✅ Model loaded successfully from Unity Catalog!")

features_df = pipeline.transform(live_df)
predictions = model.transform(features_df)

In [0]:
from pyspark.sql import functions as F

extract_risk_score = F.udf(lambda x: float(x[1]), "float")

final_report = predictions.withColumn("risk_score_pct", F.round(extract_risk_score("probability") * 100, 2)) \
                          .select("time", "temperature_2m", "risk_score_pct", "prediction")

display(final_report)

In [0]:
final_report.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("workspace.default.daily_risk_forecasts")

print("Forecast successfully saved to the database.")